# Step 5: build candidate features

Read candidates from notebook 04 and each training trip's `.nodes`, `.arcs` and `.track`. Build geometry, direction, movement, neighbour and road-network features, then save **one Parquet file**.

Features use only GPS observations, road geometry and geometric candidate ordering. Labels are carried through unchanged, including missing labels. No reference routes or point labels are read for feature calculations.

Process one complete trip at a time to avoid holding all 11.68 million rows in memory.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import shapely
from pyproj import Transformer
import pyarrow as pa
import pyarrow.parquet as pq

train = Path("Student folder/train")
candidate_folder = Path("outputs/candidates")
output = Path("outputs/features")
output.mkdir(parents=True, exist_ok=True)
metadata_columns = ["trip_id", "group_id", "point_id", "segment_id", "label"]
stationary_mps = 0.5

## 1. Movement and direction

Use projected coordinates in metres. Speed uses the previous point; acceleration uses the change in speed divided by the current time gap. Nonpositive time gaps stay missing.

Heading uses the direction from the previous GPS point to the next. It is missing at the trip ends, below **0.5 m/s**, when the two neighbouring positions are less than 1 m apart, or when the next time gap is invalid. These are starting reliability cutoffs, not tuned values.

Bearings use grid north in the same UTM projection. Opposite directions differ by 180 degrees. The raw network has reverse arc pairs in all 80 trips, so keep directed arcs separate.

In [2]:
def motion(gps_xy, time):
    gaps = np.r_[np.nan, np.diff(time)]
    gaps[gaps <= 0] = np.nan
    speed = np.r_[np.nan, np.linalg.norm(np.diff(gps_xy, axis=0), axis=1)] / gaps
    acceleration = np.r_[np.nan, np.diff(speed)] / gaps
    heading = np.full(len(time), np.nan)
    delta = gps_xy[2:] - gps_xy[:-2]
    heading[1:-1] = np.degrees(np.arctan2(delta[:, 0], delta[:, 1])) % 360
    unreliable = ~np.isfinite(speed) | (speed < stationary_mps)
    unreliable[1:-1] |= (np.linalg.norm(delta, axis=1) < 1) | ~np.isfinite(gaps[2:])
    heading[unreliable] = np.nan
    return gaps, speed, acceleration, heading


def angle_difference(a, b):
    return np.abs((a - b + 180) % 360 - 180)

## 2. Candidate features

- **Geometry:** distance, projected fraction along the segment (0–1), segment length.
- **Direction:** heading difference (0–180 degrees); zero-length segments have missing headings and fractions.
- **Movement:** speed, acceleration and previous time gap.
- **Relative features:** distance rank, distance above the nearest candidate, candidate count and heading-difference rank. Distance ties use segment ID; heading ties share their minimum rank.
- **Context:** distance and heading difference against points at t−2, t−1, t+1 and t+2, evaluated directly against the current candidate segment. Trip boundaries stay missing.
- **Connections:** previous nearest segment must end at the current segment's start; the current segment must end at the next nearest segment's start. Staying on the same segment also counts. Missing neighbouring shortlists stay missing.
- **Network:** number of directed arc rows touching each endpoint, counting each self-loop once.

The saved candidate table does not contain `rank`, so recompute it from geometric ordering. Store predictor columns as float32 to keep memory and file sizes smaller. Missing values remain missing. Because context uses future GPS observations, these features are for whole-trip matching, not immediate streaming predictions.

In [3]:
def add_features(candidates, nodes_xy, arc_nodes, gps_xy, time):
    features = candidates.sort_values(["group_id", "distance", "segment_id"]).reset_index(drop=True).copy()
    p = features["point_id"].to_numpy()
    s = features["segment_id"].to_numpy()
    start, end = arc_nodes.T
    segments = shapely.linestrings(nodes_xy[arc_nodes])
    gps = shapely.points(gps_xy)
    candidate_segments = segments[s]
    length = shapely.length(segments)
    delta = nodes_xy[end] - nodes_xy[start]
    segment_heading = np.degrees(np.arctan2(delta[:, 0], delta[:, 1])) % 360
    segment_heading[length == 0] = np.nan
    gaps, speed, acceleration, heading = motion(gps_xy, time)

    np.testing.assert_allclose(shapely.distance(candidate_segments, gps[p]), features["distance"], atol=1e-6, rtol=0)
    along = shapely.line_locate_point(candidate_segments, gps[p])
    features["segment_fraction"] = np.divide(along, length[s], out=np.full(len(p), np.nan), where=length[s] > 0)
    features["segment_length_m"] = length[s]
    features["heading_diff_deg"] = angle_difference(heading[p], segment_heading[s])
    features["speed_mps"] = speed[p]
    features["acceleration_mps2"] = acceleration[p]
    features["time_gap_s"] = gaps[p]

    groups = features.groupby("group_id", sort=False)
    features["distance_rank"] = groups.cumcount() + 1
    features["distance_above_nearest_m"] = features["distance"] - groups["distance"].transform("min")
    features["candidate_count"] = groups["segment_id"].transform("size")
    features["heading_diff_rank"] = groups["heading_diff_deg"].rank(method="min")

    for shift, name in [(-2, "prev2"), (-1, "prev"), (1, "next"), (2, "next2")]:
        neighbour = p + shift
        valid = (neighbour >= 0) & (neighbour < len(time))
        distances = np.full(len(p), np.nan)
        differences = np.full(len(p), np.nan)
        distances[valid] = shapely.distance(candidate_segments[valid], gps[neighbour[valid]])
        differences[valid] = angle_difference(heading[neighbour[valid]], segment_heading[s[valid]])
        features[f"distance_{name}_m"] = distances
        features[f"heading_diff_{name}_deg"] = differences

    nearest = np.full(len(time), -1, dtype=np.int64)
    first = features.drop_duplicates("point_id")
    nearest[first["point_id"]] = first["segment_id"]
    for shift, name in [(-1, "prev"), (1, "next")]:
        neighbour = p + shift
        valid = (neighbour >= 0) & (neighbour < len(time))
        road = np.full(len(p), -1, dtype=np.int64)
        road[valid] = nearest[neighbour[valid]]
        valid &= road >= 0
        connected = np.full(len(p), np.nan)
        if shift == -1:
            touches = end[road[valid]] == start[s[valid]]
        else:
            touches = end[s[valid]] == start[road[valid]]
        connected[valid] = (road[valid] == s[valid]) | touches
        features[f"connects_{name}"] = connected

    degree = np.bincount(arc_nodes.ravel(), minlength=len(nodes_xy))
    degree -= np.bincount(start[start == end], minlength=len(nodes_xy))
    features["start_degree"] = degree[start[s]]
    features["end_degree"] = degree[end[s]]
    columns = [c for c in features.columns if c not in metadata_columns]
    features[columns] = features[columns].astype("float32")
    return features

## 3. Build and save one trip at a time

Read complete trips using the row counts saved by notebook 04. This keeps each GPS group intact while calculating relative features. Append trips in their original order to `features.parquet`; never filter ranker groups here.

Check that every metadata field and label survives unchanged, distance calculations agree with notebook 04, groups remain sorted, fractions and angles are in range, and no feature contains infinity. The summary confirms reverse arcs and records missing labels. `feature_columns.json` lists predictors only; it excludes all IDs and the target.

In [4]:
trip_summary = pd.read_csv(candidate_folder / "summary.csv", dtype={"trip_id": str})
dtypes = {"trip_id": str, "group_id": "int64", "point_id": "int32", "segment_id": "int32", "label": "Int8"}
reader = pd.read_csv(candidate_folder / "candidates.csv", dtype=dtypes, iterator=True, float_precision="round_trip")
writer = None
checks = []
last_group = -1
try:
    for row in trip_summary.itertuples(index=False):
        trip_id = row.trip_id
        candidates = reader.get_chunk(row.candidate_rows)
        assert len(candidates) == row.candidate_rows and candidates["trip_id"].eq(trip_id).all()
        folder = train / trip_id
        nodes = pd.read_csv(folder / f"{trip_id}.nodes", sep=r"\s+", header=None, names=["lon", "lat"])
        arcs = pd.read_csv(folder / f"{trip_id}.arcs", sep=r"\s+", header=None).to_numpy()
        track = pd.read_csv(folder / f"{trip_id}.track", sep=r"\s+", header=None, names=["lon", "lat", "time"])
        zone = int((track["lon"].mean() + 180) // 6) + 1
        epsg = (32600 if track["lat"].mean() >= 0 else 32700) + zone
        projection = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
        nodes_xy = np.column_stack(projection.transform(nodes["lon"], nodes["lat"]))
        gps_xy = np.column_stack(projection.transform(track["lon"], track["lat"]))
        features = add_features(candidates, nodes_xy, arcs, gps_xy, track["time"].to_numpy())
        pd.testing.assert_frame_equal(features[metadata_columns], candidates[metadata_columns].reset_index(drop=True))
        assert features["group_id"].is_monotonic_increasing and features["group_id"].iloc[0] > last_group
        last_group = features["group_id"].iloc[-1]
        assert features["segment_fraction"].dropna().between(0, 1).all()
        assert features["heading_diff_deg"].dropna().between(0, 180).all()
        feature_columns = [c for c in features.columns if c not in metadata_columns]
        assert not np.isinf(features[feature_columns].to_numpy()).any()
        table = pa.Table.from_pandas(features, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(output / "features.parquet", table.schema, compression="zstd")
        writer.write_table(table, row_group_size=len(features))
        reverse = pd.MultiIndex.from_arrays(arcs[:, ::-1].T).isin(pd.MultiIndex.from_arrays(arcs.T))
        reverse &= arcs[:, 0] != arcs[:, 1]
        checks.append({"trip_id": trip_id, "rows": len(features), "missing_labels": features["label"].isna().sum(),
                       "reverse_arc_fraction": reverse.mean(), "zero_length_candidates": (features["segment_length_m"] == 0).sum()})
        print("Saved:", trip_id, "| Rows:", len(features), flush=True)
    try:
        assert reader.get_chunk(1).empty
    except StopIteration:
        pass
finally:
    reader.close()
    if writer is not None:
        writer.close()

summary = pd.DataFrame(checks)
summary.to_csv(output / "summary.csv", index=False)
(output / "feature_columns.json").write_text(json.dumps(feature_columns, indent=2) + "\n")
print("Saved feature rows:", summary["rows"].sum(), "| Predictor columns:", len(feature_columns))

Saved: 00000001 | Rows: 99504


Saved: 00000002 | Rows: 452648


Saved: 00000003 | Rows: 79473


Saved: 00000004 | Rows: 33019


Saved: 00000005 | Rows: 9267


Saved: 00000006 | Rows: 85566


... (71 outputs left out)

Saved: 00000097 | Rows: 97735


Saved: 00000098 | Rows: 161001


Saved: 00000099 | Rows: 176235


Saved feature rows: 11678902 | Predictor columns: 23


## 4. Final checks and preview

Verify the saved Parquet metadata matches the complete candidate CSV row count. Check group order and missing-label counts again while reading only the saved ID and label columns. Preserve missing feature values for the modelling notebooks: imputation and scaling belong inside their training pipelines.

Later, split by `trip_id`, then build the ranker's eligible groups and matching group sizes after each split. Context can use neighbouring GPS observations inside the same trip, but never labels.

In [5]:
saved = pq.ParquetFile(output / "features.parquet")
assert saved.metadata.num_rows == summary["rows"].sum() == trip_summary["candidate_rows"].sum()
previous = -1
null_labels = 0
for batch in saved.iter_batches(columns=["group_id", "label"]):
    data = batch.to_pandas()
    assert data["group_id"].is_monotonic_increasing and data["group_id"].iloc[0] >= previous
    previous = data["group_id"].iloc[-1]
    null_labels += data["label"].isna().sum()
assert null_labels == summary["missing_labels"].sum()
print("Verified rows:", saved.metadata.num_rows, "| Missing labels preserved:", null_labels)
display(saved.read_row_group(0).slice(0, 5).to_pandas())
display(summary.head())

Verified rows: 11678902 | Missing labels preserved: 176546


,trip_id,group_id,point_id,segment_id,distance,label,segment_fraction,segment_length_m,heading_diff_deg,speed_mps,...,distance_prev_m,heading_diff_prev_deg,distance_next_m,heading_diff_next_deg,distance_next2_m,heading_diff_next2_deg,connects_prev,connects_next,start_degree,end_degree
0,00000001,0,0,64973,0.722768,1,0.109775,293.237946,NaN,NaN,...,NaN,NaN,0.277461,0.688910,1.839765,3.524243,NaN,1.0,2.0,2.0
1,00000001,0,0,117639,15.132321,0,0.512692,1020.438110,NaN,NaN,...,NaN,NaN,16.079744,179.241730,13.902872,176.406403,NaN,0.0,2.0,2.0
2,00000001,0,0,2641,32.198444,0,1.000000,425.050293,NaN,NaN,...,NaN,NaN,75.811623,0.559977,125.098557,3.395309,NaN,1.0,2.0,2.0
3,00000001,0,0,50652,93.009636,0,0.845209,169.042007,NaN,NaN,...,NaN,NaN,84.054489,170.158844,77.131180,172.994171,NaN,0.0,4.0,6.0
4,00000001,0,0,53932,93.009636,0,0.154791,169.042007,NaN,NaN,...,NaN,NaN,84.054489,9.841158,77.131180,7.005826,NaN,0.0,6.0,4.0


,trip_id,rows,missing_labels,reverse_arc_fraction,zero_length_candidates
0,00000001,99504,0,0.818819,0
1,00000002,452648,28318,0.996964,0
2,00000003,79473,0,0.909807,0
3,00000004,33019,0,0.975611,0
4,00000005,9267,0,0.815706,0


References: [Shapely point projection onto lines](https://shapely.readthedocs.io/en/stable/reference/shapely.line_locate_point.html), [PyArrow Parquet writing](https://arrow.apache.org/docs/python/parquet/parquet.html).